In [ ]:
from pathlib import Path
import json
import time
import hashlib
import numpy as np
import pandas as pd
import netCDF4 as nc

# ============================================================
# 1. Inputs and outputs
# ============================================================

SOURCE = Path(
    "../data/raw/era5_land_monthly/era5_land_monthly_global_2016_2025.nc"
)

PANEL = Path("../data/study_grid_cells.csv")

HW_DIR = Path(
    "../outputs/heatwave"
)

OUT_BASE = Path(
    "../outputs/climate"
)

REF = HW_DIR / "reference_cells_all.csv"
HW = HW_DIR / "annual_local_p90_wide.csv"

KEY = ["city", "cell_id", "year"]

VARS = {
    "t2m": "K",
    "d2m": "K",
    "skt": "K",
    "sp": "Pa",
    "tp": "m",
}

MAIN = [
    "t2m_mean_C",
    "rh_mean_pct",
    "tp_annual_mm",
]

MEANS = {
    "t2m_mean_C": "t2m_C",
    "d2m_mean_C": "d2m_C",
    "skt_mean_C": "skt_C",
    "sp_mean_hPa": "sp_hPa",
    "rh_mean_pct": "rh_monthly_proxy_pct",
}


# ============================================================
# 2. Utility functions
# ============================================================

def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def stamp(path):
    s = path.stat()
    return {
        "size_bytes": s.st_size,
        "mtime_ns": s.st_mtime_ns,
    }


def as_bool(s):
    z = s.astype(str).str.lower()
    require(
        z.isin(["true", "false", "1", "0"]).all(),
        "Boolean flag has unknown values.",
    )
    return z.isin(["true", "1"])


def clean_values(a, name):
    a = np.ma.filled(
        np.ma.asarray(a, dtype="float64"),
        np.nan,
    )

    bad = np.isfinite(a) & (
        (np.abs(a) >= 1e20) | (a == -9999)
    )

    if name == "tp":
        bad |= np.isfinite(a) & (a < 0)
    else:
        bad |= np.isfinite(a) & (a <= 0)

    a[bad] = np.nan
    return a, bad


def aggregate_years(points, dates, monthly):
    frames = []

    for year in range(2016, 2026):
        ix = np.flatnonzero(dates.year == year)
        require(len(ix) == 12, f"{year} is not a complete 12 months.")

        w = dates.days_in_month.to_numpy()[ix, None]

        f = points.copy()
        f["year"] = year

        for output, source in MEANS.items():
            a = monthly[source][ix]
            complete = np.isfinite(a).all(axis=0)

            f[output + "_valid_months"] = (
                np.isfinite(a).sum(axis=0)
            )

            f[output] = np.where(
                complete,
                np.sum(a * w, axis=0) / w.sum(),
                np.nan,
            )

            f[output + "_equalmonth_diagnostic"] = np.where(
                complete,
                np.mean(a, axis=0),
                np.nan,
            )

        a = monthly["tp_month_total_mm"][ix]

        f["tp_valid_months"] = np.isfinite(a).sum(axis=0)
        f["tp_annual_mm"] = np.sum(a, axis=0)

        a = monthly["t2m_C"][ix]
        f["t2m_monthly_mean_max_C"] = np.max(a, axis=0)
        f["t2m_monthly_mean_min_C"] = np.min(a, axis=0)

        f["rh_monthly_proxy_max_pct"] = np.max(
            monthly["rh_monthly_proxy_pct"][ix],
            axis=0,
        )

        f["climate_main_complete"] = (
            f[MAIN].notna().all(axis=1)
        )

        f["climate_all_complete"] = (
            f[list(MEANS) + ["tp_annual_mm"]]
            .notna()
            .all(axis=1)
        )

        frames.append(f)

    return pd.concat(frames, ignore_index=True)


# ============================================================
# 3. Execution functions
# ============================================================

def run_climate():
    for p in [SOURCE, PANEL, REF, HW]:
        require(p.is_file(), f"Input file not found: {p}")

    before = {
        str(p): stamp(p)
        for p in [SOURCE, PANEL, REF, HW]
    }

    grid = pd.read_csv(PANEL)
    points = pd.read_csv(REF)

    require(
        not points.duplicated(["city", "cell_id"]).any(),
        "Duplicate study grid cells.",
    )

    require(
        len(points) == 652 and points.city.nunique() == 35,
        "Study area is not 35 cities and 652 grid cells.",
    )

    require(
        np.isfinite(points[["lat", "lon"]].to_numpy()).all(),
        "Grid-cell coordinates are missing.",
    )

    points = points[
        ["city", "cell_id", "lat", "lon", "baseline_eligible"]
    ].copy()

    points["baseline_eligible"] = as_bool(
        points.baseline_eligible
    )

    points = (
        points.sort_values(["city", "cell_id"])
        .reset_index(drop=True)
    )

    require(
        len(grid) == 652
        and not grid.duplicated(["city", "cell_id"]).any(),
        "Study grid-cell table is not 652 unique grid cells.",
    )

    check = grid.merge(
        points,
        on=["city", "cell_id"],
        how="outer",
        suffixes=("_grid", "_ref"),
        validate="one_to_one",
        indicator=True,
    )

    require(
        check._merge.eq("both").all(),
        "Grid-cell lists of the study grid-cell table and the heatwave results are inconsistent.",
    )

    for col in ["lat", "lon"]:
        require(
            np.allclose(
                check[col + "_grid"],
                check[col + "_ref"],
                atol=2e-6,
                rtol=0,
            ),
            f"{col} differs between the study grid-cell table and the heatwave study grid cells.",
        )

    out = OUT_BASE

    out.mkdir(parents=True, exist_ok=True)
    (out / "FAILED.txt").unlink(missing_ok=True)
    print("Output folder:", out, flush=True)

    try:
        # ----------------------------------------------------
        # 3.1 Check the original file and build the spatial correspondence
        # ----------------------------------------------------
        with nc.Dataset(SOURCE) as ds:
            history = str(getattr(ds, "history", ""))

            require(
                "moda" in history,
                "stream=moda not confirmed; this precipitation conversion cannot be applied.",
            )

            tv = ds.variables["valid_time"]

            decoded_dates = nc.num2date(
                tv[:],
                tv.units,
                calendar=getattr(tv, "calendar", "standard"),
                only_use_cftime_datetimes=False,
            )

            dates = pd.DatetimeIndex(
                pd.to_datetime(decoded_dates.astype(str))
            )

            expected = pd.date_range(
                "2016-01-01",
                "2025-12-01",
                freq="MS",
            )

            require(
                dates.equals(expected),
                "Months in the original file are not a continuous, complete 2016–2025 sequence of month starts.",
            )

            lat = np.asarray(
                ds.variables["latitude"][:],
                dtype=float,
            )

            lon = np.asarray(
                ds.variables["longitude"][:],
                dtype=float,
            )

            require(
                np.allclose(
                    np.abs(np.diff(lat)),
                    0.1,
                    atol=1e-6,
                    rtol=0,
                )
                and np.allclose(
                    np.diff(lon),
                    0.1,
                    atol=1e-6,
                    rtol=0,
                ),
                "Not the expected 0.1-degree grid.",
            )

            require(
                lon.min() >= 0 and lon.max() < 360,
                "Longitude is not in the expected 0–360 range.",
            )

            qlat = points.lat.round(6).to_numpy()
            qlon = points.lon.round(6).to_numpy() % 360

            iy = pd.Index(lat).get_indexer(
                qlat,
                method="nearest",
            )

            ix = pd.Index(lon).get_indexer(
                qlon,
                method="nearest",
            )

            dy = np.abs(lat[iy] - qlat)
            dx = np.abs((lon[ix] - qlon + 180) % 360 - 180)

            require(
                (dy <= 0.050001).all()
                and (dx <= 0.050001).all(),
                "Nearest-grid offset abnormal.",
            )

            mapping = points.copy()
            mapping["source_lat"] = lat[iy]
            mapping["source_lon"] = (lon[ix] + 180) % 360 - 180
            mapping["lat_offset_deg"] = dy
            mapping["lon_offset_deg"] = dx
            mapping["source_ilat"] = iy
            mapping["source_ilon"] = ix

            mapping.to_csv(
                out / "grid_mapping.csv",
                index=False,
                encoding="utf-8-sig",
            )

            # ------------------------------------------------
            # 3.2 Read the original file in chunks to avoid repeatedly decompressing the large file point by point
            # ------------------------------------------------
            raw = {}
            invalid = {}
            metadata = {}

            for name, units in VARS.items():
                v = ds.variables[name]

                require(
                    v.dimensions == (
                        "valid_time",
                        "latitude",
                        "longitude",
                    ),
                    f"{name}: dimensions abnormal.",
                )

                require(
                    v.units == units,
                    f"{name}: units abnormal: {v.units}",
                )

                chunks = v.chunking()

                if isinstance(chunks, (list, tuple)):
                    ct, cy, cx = chunks
                else:
                    ct, cy, cx = 12, 32, 32

                metadata[name] = {
                    "units": v.units,
                    "chunks": chunks,
                    "long_name": getattr(v, "long_name", name),
                }

                groups = {}

                for j, (y, x) in enumerate(zip(iy, ix)):
                    groups.setdefault(
                        (y // cy, x // cx),
                        [],
                    ).append(j)

                a = np.full(
                    (len(dates), len(points)),
                    np.nan,
                )

                for bi, ((by, bx), js) in enumerate(
                    groups.items(), 1
                ):
                    js = np.asarray(js)

                    y0 = int(iy[js].min())
                    x0 = int(ix[js].min())
                    y1 = int(iy[js].max() + 1)
                    x1 = int(ix[js].max() + 1)

                    for t0 in range(0, len(dates), ct):
                        t1 = min(t0 + ct, len(dates))

                        selection = (
                            slice(t0, t1),
                            slice(y0, y1),
                            slice(x0, x1),
                        )

                        for attempt in range(3):
                            try:
                                if attempt == 0:
                                    block = v[selection]
                                else:
                                    with nc.Dataset(SOURCE) as retry:
                                        block = retry.variables[name][
                                            selection
                                        ]

                                sampled = block[
                                    :,
                                    iy[js] - y0,
                                    ix[js] - x0,
                                ]

                                a[t0:t1, js] = np.ma.filled(
                                    sampled,
                                    np.nan,
                                )
                                break

                            except (OSError, RuntimeError):
                                if attempt == 2:
                                    raise
                                time.sleep(2 * (attempt + 1))

                    print(
                        f"[{name}] spatial chunk {bi}/{len(groups)}",
                        flush=True,
                    )

                raw[name], invalid[name] = clean_values(
                    a, name
                )

        # ----------------------------------------------------
        # 3.3 Unit conversion and monthly humidity derivation
        # ----------------------------------------------------
        monthly = {
            n + "_C": raw[n] - 273.15
            for n in ["t2m", "d2m", "skt"]
        }

        monthly["sp_hPa"] = raw["sp"] / 100

        monthly["tp_month_total_mm"] = (
            raw["tp"]
            * 1000
            * dates.days_in_month.to_numpy()[:, None]
        )

        t = monthly["t2m_C"]
        td = monthly["d2m_C"]

        with np.errstate(
            over="ignore",
            divide="ignore",
            invalid="ignore",
        ):
            rh = 100 * np.exp(
                17.625 * td / (243.04 + td)
                - 17.625 * t / (243.04 + t)
            )

        monthly["rh_from_monthly_raw_pct"] = rh.copy()

        rh_bad = (
            (rh < 0)
            | (rh > 100)
            | (
                ~np.isfinite(rh)
                & np.isfinite(t)
                & np.isfinite(td)
            )
        )

        monthly["rh_monthly_proxy_pct"] = np.where(
            rh_bad,
            np.nan,
            rh,
        )

        mf = pd.concat(
            [
                points.assign(
                    date=d.strftime("%Y-%m-%d")
                )
                for d in dates
            ],
            ignore_index=True,
        )

        for name, a in monthly.items():
            mf[name] = a.ravel()

        for name, bad in invalid.items():
            mf[name + "_invalid_value"] = bad.ravel()

        mf["rh_invalid_value"] = rh_bad.ravel()

        mf.to_csv(
            out / "climate_monthly_2016_2025.csv",
            index=False,
            encoding="utf-8-sig",
        )

        # ----------------------------------------------------
        # 3.4 Annual aggregation, matched to the heatwave validity flags
        # ----------------------------------------------------
        annual = aggregate_years(
            points,
            dates,
            monthly,
        )

        hw = pd.read_csv(
            HW,
            usecols=KEY + ["annual_complete"],
        )

        require(
            not hw.duplicated(KEY).any(),
            "Duplicate keys in the annual heatwave results.",
        )

        hw["heatwave_annual_complete"] = as_bool(
            hw.pop("annual_complete")
        )

        annual = annual.merge(
            hw,
            on=KEY,
            how="outer",
            validate="one_to_one",
            indicator=True,
        )

        require(
            annual._merge.eq("both").all(),
            "Annual keys of climate and heatwave results are inconsistent.",
        )

        annual = (
            annual.drop(columns="_merge")
            .sort_values(KEY)
            .reset_index(drop=True)
        )

        annual["needs_climate_review_with_valid_heatwave"] = (
            annual.heatwave_annual_complete
            & ~annual.climate_main_complete
        )

        paper = annual.loc[
            annual.year.between(2016, 2023)
        ].copy()

        require(
            len(annual) == 6520 and len(paper) == 5216,
            "Annual output row count abnormal.",
        )

        require(
            not annual.duplicated(KEY).any(),
            "Duplicate annual output keys.",
        )

        require(
            annual.loc[
                annual.tp_valid_months.lt(12),
                "tp_annual_mm",
            ].isna().all(),
            "Annual precipitation total was output despite missing months.",
        )

        annual.to_csv(
            out / "climate_annual_2016_2025.csv",
            index=False,
            encoding="utf-8-sig",
        )

        paper.to_csv(
            out / "climate_annual_2016_2023.csv",
            index=False,
            encoding="utf-8-sig",
        )

        annual.loc[
            ~annual.climate_all_complete
        ].to_csv(
            out / "missing_climate_all_years.csv",
            index=False,
            encoding="utf-8-sig",
        )

        paper.loc[
            paper.needs_climate_review_with_valid_heatwave
        ].to_csv(
            out / "paper_valid_heatwave_missing_climate.csv",
            index=False,
            encoding="utf-8-sig",
        )

        city = paper.groupby("city").agg(
            rows=("cell_id", "size"),
            study_cells=("cell_id", "nunique"),
            heatwave_complete_rows=(
                "heatwave_annual_complete",
                "sum",
            ),
            climate_complete_rows=(
                "climate_main_complete",
                "sum",
            ),
            heatwave_valid_climate_missing_rows=(
                "needs_climate_review_with_valid_heatwave",
                "sum",
            ),
        )

        city.to_csv(
            out / "city_quality_2016_2023.csv",
            encoding="utf-8-sig",
        )

        # ----------------------------------------------------
        # 3.5 Output validation and notes
        # ----------------------------------------------------
        saved = pd.read_csv(
            out / "climate_annual_2016_2025.csv"
        )

        require(
            saved[KEY].equals(annual[KEY]),
            "Keys changed after writing CSV.",
        )

        require(
            np.allclose(
                saved[MAIN],
                annual[MAIN],
                equal_nan=True,
                rtol=1e-12,
                atol=1e-10,
            ),
            "Main metrics changed after writing CSV.",
        )

        require(
            all(
                stamp(Path(p)) == s
                for p, s in before.items()
            ),
            "Input files changed during the run.",
        )

        summary = {
            "status": "completed",
            "source": str(SOURCE),
            "history": history,
            "source_files": before,
            "variable_metadata": metadata,
            "reference_sha256": hashlib.sha256(
                REF.read_bytes()
            ).hexdigest(),
            "cities": 35,
            "cells": 652,
            "annual_rows": len(annual),
            "paper_rows": len(paper),
            "main_climate_missing_paper_rows": int(
                (~paper.climate_main_complete).sum()
            ),
            "heatwave_valid_climate_missing_paper_rows": int(
                paper.needs_climate_review_with_valid_heatwave.sum()
            ),
            "invalid_raw_values": {
                k: int(v.sum())
                for k, v in invalid.items()
            },
            "invalid_rh_months": int(rh_bad.sum()),
        }

        (out / "summary.json").write_text(
            json.dumps(
                summary,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

        print("Output folder:", out)

        return out, annual, city

    except (Exception, KeyboardInterrupt) as exc:
        (out / "FAILED.txt").write_text(
            repr(exc),
            encoding="utf-8",
        )
        raise


# ============================================================
# 4. Run
# ============================================================

(
    CLIMATE_OUT_DIR,
    CLIMATE_ANNUAL,
    CLIMATE_CITY_QUALITY,
) = run_climate()

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import xarray as xr

HEATWAVE_RUN = Path("../outputs/heatwave")
OUT_BASE = Path("../outputs/cdd")

BASES_C = [22.0]
YEARS = list(range(2016, 2026))
PAPER_YEARS = list(range(2016, 2024))
EXPECTED_POINTS = 652
KEY = ["city", "cell_id", "year"]


def require(ok, message):
    if not bool(ok):
        raise RuntimeError(message)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def load_inputs():
    summary = json.loads((HEATWAVE_RUN / "summary.json").read_text(encoding="utf-8"))
    manifest_path = Path(summary["input_manifest"])
    require(manifest_path.is_file(), f"Heatwave temperature input manifest not found: {manifest_path}")
    require(sha256(manifest_path) == summary["input_manifest_sha256"], "Temperature input manifest does not match the heatwave run record")
    manifest = pd.read_csv(manifest_path)
    require(manifest.city.is_unique and len(manifest) == 35, "Temperature input manifest is not one file per city for 35 cities")
    points = pd.read_csv(HEATWAVE_RUN / "reference_cells_all.csv", usecols=["city", "cell_id", "lat", "lon"])
    require(len(points) == EXPECTED_POINTS and not points.duplicated(["city", "cell_id"]).any(), "Study grid-cell list abnormal")
    return manifest, points, manifest_path


def city_cdd(city, nc_path, expected_sha, pts):
    require(sha256(nc_path) == expected_sha, f"{city}: checksum of temperature file does not match the manifest")
    with xr.open_dataset(nc_path) as ds:
        require(set(ds.cell_id.values.tolist()) == set(pts.cell_id), f"{city}: grid cells of the temperature file do not match the study grid cells")
        ds = ds.sel(cell_id=sorted(pts.cell_id))
        require(np.allclose(ds.lat.values, pts.sort_values("cell_id").lat, atol=1e-6) and
                np.allclose(ds.lon.values, pts.sort_values("cell_id").lon, atol=1e-6), f"{city}: grid-cell coordinates do not match")
        time = pd.DatetimeIndex(ds.time.values)
        require(time.equals(pd.date_range("2016-01-01", "2025-12-31", freq="D")), f"{city}: dates are not a complete 2016–2025")
        tx = ds.tx_C.values.astype("float64")
        tn = ds.tn_C.values.astype("float64")
        valid = ds.pair_valid.values.astype(bool) & np.isfinite(tx) & np.isfinite(tn)
        cells = ds.cell_id.values
    tmean = np.where(valid, (tx + tn) / 2.0, np.nan)
    rows = []
    for y in YEARS:
        ix = np.flatnonzero(time.year == y)
        v = valid[ix]
        n_valid = v.sum(axis=0)
        complete = n_valid == len(ix)
        out = dict(city=city, cell_id=cells, year=y, days_in_year=len(ix), valid_days=n_valid, cdd_complete=complete,
                   tmean_annual_C=np.where(complete, np.nanmean(tmean[ix], axis=0) if v.any() else np.nan, np.nan))
        for b in BASES_C:
            s = np.nansum(np.clip(tmean[ix] - b, 0, None), axis=0)
            out[f"cdd_{b:g}C"] = np.where(complete, s, np.nan)
        rows.append(pd.DataFrame(out))
    return pd.concat(rows, ignore_index=True)


def run_cdd():
    manifest, points, manifest_path = load_inputs()
    out_dir = OUT_BASE
    out_dir.mkdir(parents=True, exist_ok=True)

    parts = []
    for r in manifest.sort_values("city").itertuples():
        parts.append(city_cdd(r.city, manifest_path.parent / r.output_nc, r.output_sha256, points[points.city == r.city]))
        print(f"{r.city:32s} done", flush=True)
    cell = pd.concat(parts, ignore_index=True).sort_values(KEY).reset_index(drop=True)
    require(len(cell) == EXPECTED_POINTS * len(YEARS) and not cell.duplicated(KEY).any(), "Grid-cell-year output row count or keys abnormal")

    agg = {"n_cells": ("cell_id", "size"), "n_cells_complete": ("cdd_complete", "sum")}
    for b in BASES_C:
        agg[f"cdd_{b:g}C_city"] = (f"cdd_{b:g}C", "mean")
        agg[f"cdd_{b:g}C_city_sd"] = (f"cdd_{b:g}C", "std")
    city = cell.groupby(["city", "year"]).agg(**agg).reset_index()
    city["city_complete_share"] = city.n_cells_complete / city.n_cells
    require(city.loc[city.n_cells_complete == 0, "cdd_22C_city"].isna().all(), "A city-year with no valid grid cells must not have a city CDD")

    panel = cell.merge(city[["city", "year", "cdd_22C_city", "n_cells_complete", "city_complete_share"]],
                       on=["city", "year"], how="left", validate="many_to_one")
    paper = panel[panel.year.isin(PAPER_YEARS)].reset_index(drop=True)
    require(len(paper) == EXPECTED_POINTS * len(PAPER_YEARS), "2016–2023 panel row count abnormal")

    for name, df in [("cdd_cell_2016_2025.csv", cell), ("cdd_city_2016_2025.csv", city),
                     ("cdd_panel_2016_2023.csv", paper)]:
        df.to_csv(out_dir / name, index=False, encoding="utf-8-sig", float_format="%.17g")
    back = pd.read_csv(out_dir / "cdd_panel_2016_2023.csv", float_precision="round_trip")
    pd.testing.assert_frame_equal(paper, back, check_dtype=False, check_exact=True)

    summary = dict(
        status="calculation_complete",
        temperature_input_manifest=str(manifest_path), heatwave_run=str(HEATWAVE_RUN),
        cell_year_rows=int(len(cell)), cell_year_complete=int(cell.cdd_complete.sum()),
        paper_rows=int(len(paper)), paper_cell_complete=int(paper.cdd_complete.sum()),
        paper_city_cdd_missing_rows=int(paper.cdd_22C_city.isna().sum()))
    (out_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

    print("Output folder:", out_dir)
    return paper, city


CDD_PANEL, CDD_CITY = run_cdd()